In [1]:
import duckdb
import pandas as pd 
import numpy as np


In [2]:
#Con EDA buscaremos las 5 rutas más utilizadas
query_top_rutas  = """ 
    SELECT
        TRIM(UPPER(s.seller_state)) AS origen,
        TRIM(UPPER(c.customer_state)) AS destino,
        COUNT(DISTINCT o.order_id) AS total_pedido,
        SUM(p.product_length_cm * p.product_height_cm * p.product_width_cm) AS volumen_historico_cm3

    FROM '../data/olist_orders_dataset.csv' AS o 
    JOIN '../data/olist_order_items_dataset.csv' AS i ON o.order_id = i.order_id
    JOIN '../data/olist_customers_dataset.csv' AS c ON o.customer_id = c.customer_id
    JOIN '../data/olist_sellers_dataset.csv' AS s ON i.seller_id = s.seller_id
    JOIN '../data/olist_products_dataset.csv' AS p ON i.product_id = p.product_id

    WHERE TRIM(UPPER(o.order_status)) NOT IN ('CANCELED', 'UNAVAILABLE')

    GROUP BY origen, destino 
    ORDER BY volumen_historico_cm3 DESC
    LIMIT 5;
"""

df_top_rutas = duckdb.query(query_top_rutas).df()
df_top_rutas

,origen,destino,total_pedido,volumen_historico_cm3
0,SP,SP,31293,556239440.0
1,SP,RJ,8421,158391337.0
2,SP,MG,7565,131652907.0
3,SP,RS,3661,67602011.0
4,SP,PR,3167,56111834.0


In [3]:
rutas_criticas = (df_top_rutas['origen'] + "-" + df_top_rutas['destino']).tolist()
print(f"Las rutas que nuestro modelo va a optimizar son: {rutas_criticas}")

Las rutas que nuestro modelo va a optimizar son: ['SP-SP', 'SP-RJ', 'SP-MG', 'SP-RS', 'SP-PR']


In [4]:

rutas_sql = "', '".join(rutas_criticas)

query_final = f""" 
    WITH promedio_categoria AS (
        SELECT
            product_category_name,
            AVG(product_length_cm * product_height_cm * product_width_cm) AS volumen_promedio,
            AVG(product_weight_g) AS promedio_peso
        FROM '../data/olist_products_dataset.csv'
        GROUP BY product_category_name    
    ),
    promedio_global AS (
        SELECT
            AVG(product_length_cm * product_height_cm * product_width_cm) AS vol_global,
            AVG(product_weight_g) AS peso_global
        FROM '../data/olist_products_dataset.csv'
    ),
    pedidos_rutas_top AS (
        SELECT
            o.order_delivered_carrier_date::DATE AS fecha,
            (TRIM(UPPER(s.seller_state)) || '-' || TRIM(UPPER(c.customer_state))) AS ruta,
            p.product_category_name,
            p.product_length_cm,
            p.product_height_cm,
            p.product_width_cm,
            p.product_weight_g
        FROM '../data/olist_orders_dataset.csv' AS o
        JOIN '../data/olist_order_items_dataset.csv' AS i ON o.order_id = i.order_id
        JOIN '../data/olist_customers_dataset.csv' AS c ON o.customer_id = c.customer_id
        JOIN '../data/olist_products_dataset.csv' AS p ON i.product_id = p.product_id
        JOIN '../data/olist_sellers_dataset.csv' AS s ON i.seller_id = s.seller_id
        WHERE TRIM(UPPER(o.order_status)) NOT IN ('CANCELED', 'UNAVAILABLE')    
          AND o.order_delivered_carrier_date IS NOT NULL    
          AND (TRIM(UPPER(s.seller_state)) || '-' || TRIM(UPPER(c.customer_state))) IN ('{rutas_sql}')
    )
    SELECT
        p_top.fecha,
        p_top.ruta,
        COUNT(*) AS total_paquetes_diarios,
        SUM(
            COALESCE(
                NULLIF(p_top.product_length_cm * p_top.product_height_cm * p_top.product_width_cm, 0),
                pc.volumen_promedio,
                pg.vol_global 
            )
        ) AS volumen_total_cm3,
        SUM(
            COALESCE(
                NULLIF(p_top.product_weight_g, 0),
                pc.promedio_peso,
                pg.peso_global
            )
        ) AS peso_total_g
    FROM pedidos_rutas_top AS p_top
    LEFT JOIN promedio_categoria AS pc ON p_top.product_category_name = pc.product_category_name
    CROSS JOIN promedio_global AS pg
    GROUP BY p_top.fecha, p_top.ruta
    ORDER BY p_top.fecha, p_top.ruta;
"""

df_final = duckdb.query(query_final).df()
df_final

,fecha,ruta,total_paquetes_diarios,volumen_total_cm3,peso_total_g
0,2016-10-08,SP-SP,1,2640.0,9000.0
1,2016-10-09,SP-PR,4,24288.0,5000.0
2,2016-10-10,SP-MG,2,7665.0,743.0
3,2016-10-10,SP-PR,1,4641.0,443.0
4,2016-10-10,SP-RJ,3,18644.0,1700.0
...,...,...,...,...,...
2296,2018-08-30,SP-SP,1,3960.0,363.0
2297,2018-08-31,SP-SP,5,73377.0,8850.0
2298,2018-09-03,SP-RS,1,55900.0,14813.0
2299,2018-09-04,SP-SP,1,34816.0,10400.0


In [5]:

df_final['fecha'] = pd.to_datetime(df_final['fecha'])
df_pivot = df_final.pivot_table(
    index = 'fecha',
    columns = 'ruta',
    values= ['total_paquetes_diarios','volumen_total_cm3','peso_total_g'],
    aggfunc ='sum'
)
df_pivot = df_pivot.asfreq('D', fill_value=0)
df_final_completo = df_pivot.stack(level=1,future_stack=True).reset_index()
df_final_completo = df_final_completo.fillna(0)

df_final_completo
df_final_completo.to_csv('../data/dataset_rutas_limpios.csv', index=False)
